# Análise de logs de acesso (HydraSensor)

Este notebook lê o CSV exportado pelo backend (mesmas colunas que `GET /v1/access-events/export.csv` ou o arquivo `rfid_access_log.csv`) e responde às perguntas do relatório usando **Pandas**.

**Antes de rodar:** na pasta `analise_dados`, execute `python exportar_csv.py` (cópia local) ou `python exportar_csv.py --api` com o Flask ativo. Se não houver `dados/access_events.csv`, o notebook usa `dados/exemplo_access_events.csv`.

In [1]:
from pathlib import Path

import pandas as pd

BASE = Path.cwd()
if BASE.name != "analise_dados":
    candidato = BASE / "analise_dados"
    if candidato.is_dir():
        BASE = candidato

CSV_EXPORT = BASE / "dados" / "access_events.csv"
#CSV_EXEMPLO = BASE / "dados" / "exemplo_access_events.csv"

if CSV_EXPORT.is_file() and CSV_EXPORT.stat().st_size > 50:
    origem = CSV_EXPORT
else:
    origem = CSV_EXPORT
    print("Aviso: usando CSV de exemplo. Rode exportar_csv.py para dados reais.")

df = pd.read_csv(origem)
df["read_at"] = pd.to_datetime(df["read_at"], errors="coerce")
df["dia"] = df["read_at"].dt.strftime("%Y-%m-%d")
df["authorized"] = df["authorized"].astype(int)

print(f"Origem: {origem}")
print(f"Registros: {len(df)}")
df.head()

Origem: C:\Users\gport\OneDrive\Área de Trabalho\HydraSensor\analise_dados\dados\access_events.csv
Registros: 31


,id,tag_id,collaborator_id,collaborator_name,registration,authorized,event_type,origin,message,read_at,created_at,synced,dia
0,1,497104650867,NaN,Desconhecido,NaN,0,invasao,raspberry-rfid,Tag 497104650867 desconhecida. Possivel tentat...,2026-05-13 18:57:33,2026-05-13T18:57:33,1,2026-05-13
1,2,497104650867,NaN,Desconhecido,NaN,0,invasao,raspberry-rfid,Tag 497104650867 desconhecida. Possivel tentat...,2026-05-13 18:57:37,2026-05-13T18:57:37,1,2026-05-13
2,3,497104650867,NaN,Desconhecido,NaN,0,invasao,raspberry-rfid,Tag 497104650867 desconhecida. Possivel tentat...,2026-05-13 18:58:31,2026-05-13T18:58:31,1,2026-05-13
3,4,497104650867,NaN,Desconhecido,NaN,0,invasao,raspberry-rfid,Tag 497104650867 desconhecida. Possivel tentat...,2026-05-13 18:59:49,2026-05-13T19:01:27,1,2026-05-13
4,5,497104650867,NaN,Desconhecido,NaN,0,invasao,raspberry-rfid,Tag 497104650867 desconhecida. Possivel tentat...,2026-05-13 18:59:58,2026-05-13T19:01:28,1,2026-05-13


In [2]:
# Parâmetros da análise (altere o dia e o colaborador conforme seus dados)
DIA_REFERENCIA = "2026-05-13"  # YYYY-MM-DD
COLABORADOR_ID = 1  # id numérico do colaborador no banco
NOME_COLABORADOR = "Ana Silva"  # apenas para exibição

d = df[df["dia"] == DIA_REFERENCIA].copy()
print(f"Eventos no dia {DIA_REFERENCIA}: {len(d)}")

Eventos no dia 2026-05-13: 31


## 1. Quantas pessoas entraram na sala no dia?

Contagem de eventos `entrada` autorizados e quantidade de **colaboradores distintos** com pelo menos uma entrada.

In [3]:
entradas = d[(d["event_type"] == "entrada") & (d["authorized"] == 1)]
total_eventos_entrada = len(entradas)
pessoas_distintas_entrada = entradas["collaborator_id"].dropna().nunique()

print(f"Total de registros de entrada (autorizados): {total_eventos_entrada}")
print(f"Colaboradores distintos que entraram: {int(pessoas_distintas_entrada)}")

Total de registros de entrada (autorizados): 6
Colaboradores distintos que entraram: 1


## 2. Quantas pessoas saíram da sala no dia?

Eventos `saida` autorizados e colaboradores distintos.

In [4]:
saidas = d[(d["event_type"] == "saida") & (d["authorized"] == 1)]
total_eventos_saida = len(saidas)
pessoas_distintas_saida = saidas["collaborator_id"].dropna().nunique()

print(f"Total de registros de saída (autorizados): {total_eventos_saida}")
print(f"Colaboradores distintos com saída registrada: {int(pessoas_distintas_saida)}")

Total de registros de saída (autorizados): 5
Colaboradores distintos com saída registrada: 1


## 3. Quanto tempo um colaborador permaneceu na sala no dia?

Emparelha `entrada` → `saida` em ordem de `read_at`. Se a última leitura do dia for `entrada` sem `saida`, o intervalo aberto vai até o **fim do dia** (23:59:59), alinhado à lógica de relatórios do backend.

In [5]:
from datetime import time


def permanencia_colaborador_no_dia(frame: pd.DataFrame, dia: str, collaborator_id: int) -> pd.Series:
    ev = frame[
        (frame["dia"] == dia)
        & (frame["collaborator_id"] == collaborator_id)
        & (frame["event_type"].isin(["entrada", "saida"]))
        & (frame["authorized"] == 1)
    ].copy()
    sort_cols = ["read_at"] + (["id"] if "id" in ev.columns else [])
    ev = ev.sort_values(sort_cols)

    fim_dia = pd.Timestamp.combine(pd.to_datetime(dia).date(), time(23, 59, 59))
    total = pd.Timedelta(0)
    aberta = None

    for _, row in ev.iterrows():
        if row["event_type"] == "entrada":
            aberta = row["read_at"]
        elif row["event_type"] == "saida" and aberta is not None:
            total += row["read_at"] - aberta
            aberta = None

    if aberta is not None:
        total += fim_dia - aberta

    return pd.Series({"total": total, "total_segundos": int(total.total_seconds())})


res = permanencia_colaborador_no_dia(df, DIA_REFERENCIA, COLABORADOR_ID)
print(f"Colaborador: {NOME_COLABORADOR} (id={COLABORADOR_ID})")
print(f"Tempo total no dia {DIA_REFERENCIA}: {res['total']} ({res['total_segundos']} s)")

Colaborador: Ana Silva (id=1)
Tempo total no dia 2026-05-13: 0 days 04:45:49 (17149 s)


## 4. Quantas tentativas de acesso negado no dia?

Eventos com `event_type == "acesso_negado"`.

In [6]:
negados = d[d["event_type"] == "acesso_negado"]
print(f"Acessos negados no dia {DIA_REFERENCIA}: {len(negados)}")

Acessos negados no dia 2026-05-13: 10


## 5. Quantas tentativas de invasão no dia?

Eventos com `event_type == "invasao"` (tag desconhecida).

In [7]:
invasoes = d[d["event_type"] == "invasao"]
print(f"Tentativas de invasão no dia {DIA_REFERENCIA}: {len(invasoes)}")

Tentativas de invasão no dia 2026-05-13: 10


## 6. Colaboradores não autorizados com mais tentativas no dia

Ranking a partir de `acesso_negado`: agrupa por `collaborator_id` (ou `tag_id` se id nulo) e ordena por número de tentativas.

In [9]:
na = d[d["event_type"] == "acesso_negado"].copy()
na["chave"] = na["collaborator_id"].fillna(na["tag_id"]).astype(str)
ranking = (
    na.groupby(["chave", "collaborator_name", "registration"], dropna=False)
    .size()
    .reset_index(name="tentativas")
    .sort_values("tentativas", ascending=False)
)
print("Ranking de tentativas (acesso negado):")
display(ranking)

Ranking de tentativas (acesso negado):


,chave,collaborator_name,registration,tentativas
0,1.0,Bernardo Antunes Heckler,1137118.0,5
